In [1]:
import requests
import json

# Define tools that can be used by the agent

In [2]:
def get_weather(city: str):
    return f"The weather in {city} is 30°C and sunny."

def celsius_to_custom(celsius):
    return f"{celsius + 10}°T"

TOOLS = {
    "get_weather": get_weather,
    "celsius_to_custom": celsius_to_custom,
}

# Prompt defining the task and tools
Note that here the definition of tools are mentioned directly in the prompt. This is usually taken care by the frameworks like crewAI, OpenAI Agents SDK, etc.

In [3]:
SYSTEM_PROMPT = """
You are an useful AI agent that help in answering user questions. You will be assited by a set of tools that you can call to get more information.

You must decide whether to:
- Respond directly
- Call a tool to get more information before responding

Available tools:
1. get_weather(city: str)- Returns the current weather for a given city.
2. celsius_to_custom(celsius: float)- Converts Celsius to a custom scale °T.


If using a tool, respond ONLY in this JSON format:

{
  "action": "tool_call",
  "tools": {"tool_name": {
    "arg1": "value",
    "arg2": "value"
    }
  }
}

If responding directly, use:

{
  "action": "final",
  "tools": [],
  "answer": "your answer"
}

Rules:
- DO NOT add extra text
- ALWAYS output valid JSON
- NEVER explain your decision
"""

In [4]:
model = "gemma4:e2b"
url = "http://localhost:11434/api/generate"

def call_llm(prompt):
    response = requests.post(
        url,
        json={
            "model": model,
            "prompt": prompt,
            "stream": False
        }
    )
    return response.json()["response"]

In [5]:
def parse_llm_output(output):
    try:
        return json.loads(output)
    except json.JSONDecodeError:
        return {
            "action": "final",
            "answer": "Failed to parse LLM output"
        }

In [ ]:
def run_agent(user_input, max_steps=5):
    print(user_input)
    context = f"{SYSTEM_PROMPT}\nUser: {user_input}\n"

    for step in range(max_steps):
        print(f"\n--- Iteration {step + 1} ---")
        llm_output = call_llm(context)
        # print(context)
        print("LLM Output:", llm_output)

        parsed = parse_llm_output(llm_output)

        if parsed["action"] == "final":
            return parsed["answer"]

        elif parsed["tools"]:
            tools = parsed["tools"]
            for tool_name in tools:
                args = tools.get(tool_name, {})
                try:
                    result = TOOLS[tool_name](**args)
                except Exception as e:
                    result = f"Tool execution failed: {str(e)}"

            print("Tool Used:", tool_name)
            print("Arguments:", args)

            context += f"""
You called a tool {tool_name} with arguments {args} and got result: {result}. 
If this is enough to answer the user's question, respond with action 'final' and provide the answer in JSON format
else use a tool.
"""

        else:
            return "Unknown action from LLM"

    return "Max steps reached without final answer."

In [ ]:
run_agent("What is the temperature in New York? can you provide that in °T?")

What is the temperature in New York? can you provide that °T?

--- Iteration 1 ---

You are an useful AI agent that help in answering user questions. You will be assited by a set of tools that you can call to get more information.

You must decide whether to:
- Respond directly
- Call a tool to get more information before responding

Available tools:
1. get_weather(city: str)- Returns the current weather for a given city.
2. celsius_to_custom(celsius: float)- Converts Celsius to a custom scale °T.


If using a tool, respond ONLY in this JSON format:

{
  "action": "tool_call",
  "tools": {"tool_name": {
    "arg1": "value",
    "arg2": "value"
    }
  }
}

If responding directly, use:

{
  "action": "final",
  "tools": [],
  "answer": "your answer"
}

Rules:
- DO NOT add extra text
- ALWAYS output valid JSON
- NEVER explain your decision

User: What is the temperature in New York? can you provide that °T?

LLM Output: {
  "action": "tool_call",
  "tools": {
    "get_weather": {
      "

'The temperature in New York is 40.0°T.'